# MVP — Previsão de Cancelamento de Clientes (Churn) em Telecom

**Autor:** Mateus M. Matos
**Disciplina:** Machine Learning & Analytics — Pós-Graduação em Ciência de Dados e Analytics
**Tipo de tarefa:** Classificação binária supervisionada

---

Este notebook é um relatório técnico executável: cada etapa traz o código, o resultado e a explicação da decisão por trás dele. Ele roda do começo ao fim sem upload, login ou token, porque os dados são lidos direto de uma URL pública.

## 1. Apresentação do problema

### O problema
Empresas de telecom perdem receita quando um cliente cancela o serviço, um fenômeno chamado de churn. Conquistar cliente novo costuma sair bem mais caro do que manter um atual, então descobrir quem está prestes a cancelar dá à empresa a chance de agir antes, com uma oferta, um contato ou um desconto, e evitar a perda.

### Objetivo do modelo
Construir um modelo que, a partir das características de um cliente (tempo de casa, tipo de contrato, serviços contratados, forma de pagamento e afins), preveja se ele vai cancelar (`Yes`) ou permanecer (`No`).

### Variável-alvo
A coluna `Churn`, que assume dois valores: `Yes` para quem cancelou e `No` para quem permaneceu.

### Por que isso é um problema de Machine Learning?
Porque a ideia é aprender um padrão a partir de exemplos passados. Temos milhares de clientes com o desfecho já conhecido e suas características, e o modelo aprende a relação entre um e outro para depois prever o desfecho de clientes novos, ainda sem rótulo. Como a resposta é uma categoria (`Yes` ou `No`), o problema é de classificação, e binária, já que só há duas classes possíveis.

### Premissas, hipóteses e restrições
- Premissa: os dados históricos representam bem o comportamento futuro dos clientes, ou seja, o padrão de cancelamento não muda de forma radical de um período para o outro.
- Hipótese: características como tipo de contrato, tempo de casa (`tenure`) e forma de pagamento têm relação com a chance de cancelamento.
- Restrição: trabalho só com os atributos que existem no dataset. Não há dados externos, como reclamações ou uso real do serviço, que poderiam enriquecer o modelo.
- Foco do MVP: o objetivo não é chegar ao melhor modelo possível, mas entregar uma solução consistente e bem justificada de ponta a ponta.

## 2. Configuração do ambiente e carregamento dos dados

Importo as bibliotecas e fixo uma semente aleatória (`SEED`). Fixar a semente é o que garante reprodutibilidade: toda vez que o notebook roda, as divisões aleatórias e os modelos caem no mesmo resultado. Sem isso, quem reproduzir o notebook veria números um pouco diferentes dos meus.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

print("Bibliotecas carregadas com sucesso.")

Carrego os dados direto da URL pública, a versão raw do GitHub. Esse é o ponto que garante a reprodutibilidade: qualquer pessoa executa o notebook sem precisar baixar nada na mão.

In [ ]:
URL = "https://raw.githubusercontent.com/MateusMMatos/mvp-churn-telco/main/Telco-Customer-Churn.csv"
df = pd.read_csv(URL)

print(f"Dataset carregado: {df.shape[0]} clientes (linhas) e {df.shape[1]} atributos (colunas).")
df.head()

## 3. Conhecendo os dados (análise exploratória)

Antes de modelar, preciso entender os dados: quantas linhas e colunas existem, o tipo de cada variável, se há valores faltando e como a variável-alvo se distribui. Pular essa etapa é o caminho mais curto para um erro grosseiro lá na frente. Vale o velho ditado da área, garbage in, garbage out: não existe modelo bom em cima de dado ruim.

### Sobre os dados: fonte, variáveis e limitações

- Fonte: dataset Telco Customer Churn, publicado pela IBM como base de amostra. É um conjunto de dados fictício de uma operadora imaginária, feito para ensino, sem clientes reais e portanto sem nenhuma questão de privacidade.
- Volume: 7.043 registros (clientes) e 21 atributos, confirmados na célula acima.
- Variável-alvo: `Churn`, que indica se o cliente cancelou (`Yes`) ou permaneceu (`No`).

Dicionário das principais variáveis:

| Variável | Significado |
|---|---|
| `customerID` | Identificador único do cliente (será removido, não tem poder preditivo) |
| `gender`, `SeniorCitizen`, `Partner`, `Dependents` | Perfil demográfico (sexo, se é idoso, se tem cônjuge/dependentes) |
| `tenure` | Nº de meses que a pessoa é cliente |
| `PhoneService`, `MultipleLines`, `InternetService` | Serviços de telefonia e internet contratados |
| `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` | Serviços adicionais (Yes/No/sem internet) |
| `Contract` | Tipo de contrato (mês a mês, 1 ano, 2 anos) |
| `PaperlessBilling`, `PaymentMethod` | Fatura digital e forma de pagamento |
| `MonthlyCharges`, `TotalCharges` | Mensalidade e gasto total acumulado |
| `Churn` | Alvo: o cliente cancelou? (`Yes`/`No`) |

Por que escolhi esta base:
- É um problema de negócio real e comum, a retenção de clientes, com aplicação prática clara.
- É uma tarefa de classificação binária, alinhada ao que estudei.
- Tem desbalanceamento de classes, o que dá material para discutir métricas.
- Tem variáveis mistas, categóricas e numéricas, o que exercita o pré-processamento.
- É pública e carrega por URL, o que atende à exigência de reprodutibilidade.
- Não foi usada nas aulas da sprint.

Limitações que já conheço do dataset:
- Por ser fictício, ele tende a ser mais comportado que dado real, com padrões possivelmente mais nítidos do que apareceria na prática.
- É uma fotografia estática. Não há informação de quando cada cliente cancelou, nem histórico de uso.
- Faltam variáveis que costumam ajudar bastante, como reclamações, satisfação e chamados de suporte.

### 3.1 Dimensões e tipos das variáveis

In [ ]:
print(f"O dataset tem {df.shape[0]} registros (clientes) e {df.shape[1]} atributos (colunas).\n")
df.info()

São 7.043 clientes e 21 colunas. A maioria, 18 delas, é texto (`object`/`str`), como `gender`, `Contract` e `PaymentMethod`. O computador não treina em cima de texto, então essas colunas vão precisar virar número na etapa de preparação.

Só 3 colunas são numéricas: `SeniorCitizen`, `tenure` (meses de contrato) e `MonthlyCharges` (mensalidade).

Há um ponto de atenção com a qualidade do dado: `TotalCharges`, o gasto total do cliente, aparece como texto quando deveria ser número. Isso quase sempre indica alguma sujeira escondida na coluna. Vou investigar e corrigir na preparação.

### 3.2 Estatísticas descritivas das variáveis numéricas

In [ ]:
df.describe().round(2)

A coluna `tenure` (tempo de casa) vai de 0 a 72 meses, com média perto de 32: há desde clientes recém-chegados até clientes com seis anos de casa.

`MonthlyCharges` (mensalidade) vai de R\$ 18,25 a R\$ 118,75.

Note que o `describe()` só trouxe 3 colunas. A `TotalCharges` ficou de fora porque o pandas a tratou como texto, e não há estatística de texto para calcular. É a confirmação do problema apontado acima.

### 3.3 Distribuição da variável-alvo (`Churn`): análise de desbalanceamento

In [ ]:
contagem = df["Churn"].value_counts()
print(contagem)
print()
print((df["Churn"].value_counts(normalize=True) * 100).round(1))

cores = {"No": "#2ecc71", "Yes": "#e74c3c"}
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(contagem.index, contagem.values,
       color=[cores.get(v, "#888888") for v in contagem.index])
ax.set_title("Distribuição do alvo (Churn)")
ax.set_xlabel("Cliente cancelou?")
ax.set_ylabel("Nº de clientes")
for i, v in enumerate(contagem.values):
    ax.text(i, v, str(v), ha="center", va="bottom")
plt.show()

Esse é um dos pontos centrais do projeto. 73,5% dos clientes permaneceram (`No`) e só 26,5% cancelaram (`Yes`), então as classes estão desbalanceadas.

E por que isso pesa tanto? Um modelo preguiçoso, que simplesmente chutasse "ninguém cancela" para todo mundo, já acertaria 73,5% sem ter aprendido nada. É por isso que a acurácia sozinha engana neste problema. Vou precisar de métricas como precisão, recall e F1-score, que enxergam o acerto na classe minoritária, os clientes que cancelam, que é justamente a que interessa para o negócio.

Essa observação vai guiar tanto a escolha do baseline quanto a das métricas de avaliação mais adiante.

### 3.4 Quais características puxam o cancelamento?

Agora vou atrás de quem cancela mais. Para cada característica, calculo a taxa de cancelamento, que é a porcentagem de clientes daquele grupo que deram churn. Isso ajuda a entender o problema e já antecipa quais variáveis o modelo provavelmente vai achar importantes.

A função abaixo calcula a taxa de churn de qualquer coluna categórica. Criei ela para não repetir o mesmo código várias vezes (boa prática, ver seção 8).

In [ ]:
def taxa_churn(coluna):
    """Retorna a % de clientes que cancelaram (Churn='Yes') em cada categoria da coluna."""
    tab = (pd.crosstab(df[coluna], df["Churn"], normalize="index")["Yes"] * 100).round(1)
    return tab.sort_values(ascending=False)

tabela_contrato = taxa_churn("Contract")
print(tabela_contrato)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(tabela_contrato.index, tabela_contrato.values, color="#c44e52")
ax.set_title("Taxa de cancelamento por tipo de contrato")
ax.set_xlabel("Tipo de contrato")
ax.set_ylabel("% que cancelou")
for i, v in enumerate(tabela_contrato.values):
    ax.text(i, v, f"{v}%", ha="center", va="bottom")
plt.show()

O tipo de contrato foi um dos fatores que mais pesou. No plano mês a mês, 42,7% dos clientes cancelam; no de 2 anos, só 2,8%. Faz sentido no mundo real: o mensal não tem fidelidade nem multa, então a pessoa sai quando quiser. Já quem fechou dois anos fica preso ao contrato, e provavelmente estava mais satisfeito quando assinou.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=df, x="tenure", hue="Churn", bins=30, element="step",
             stat="density", common_norm=False,
             palette=["#2ecc71", "#e74c3c"], ax=ax)
ax.set_title("Distribuição do tempo de casa (tenure) por churn")
ax.set_xlabel("Meses como cliente (tenure)")
plt.show()

print("Tempo de casa médio:")
print(df.groupby("Churn")["tenure"].mean().round(1))

O cancelamento se concentra nos primeiros meses. Quem cancela tem, em média, 18 meses de casa; quem permanece, 38. No gráfico, o pico do churn aparece logo no início do eixo: cliente novo é cliente em risco. Isso sustenta a estratégia de investir na retenção já nos primeiros meses.

---

Reunindo a exploração, o perfil de risco de cancelamento fica assim:

| Sinal de risco | Grupo com mais churn |
|---|---|
| Contrato | Mês a mês (42,7%) |
| Pagamento | Cheque eletrônico (45,3%) |
| Internet | Fibra óptica (41,9%) |
| Tempo de casa | Clientes novos (~18 meses) |
| Mensalidade | Mais alta (~R\$ 74) |

Essas relações ajudam a entender o problema e mostram que há sinal aprendível nos dados, o que favorece a modelagem. Nenhuma variável sozinha explica o cancelamento, então um modelo que combine várias delas tende a ir melhor.

## 4. Preparação dos dados

Agora deixo os dados prontos para a modelagem. Cada decisão daqui para frente vem justificada, porque a ideia não é só aplicar transformações e sim explicar por que cada uma faz sentido. As transformações que dependem de aprender algo dos dados, como a codificação e a padronização, vão acontecer dentro de um pipeline ajustado só no treino, para não vazar dado do teste. Volto nesse ponto nas próximas etapas.

### 4.1 Corrigindo a coluna `TotalCharges`

Na exploração eu vi que `TotalCharges`, o gasto total do cliente, veio como texto em vez de número. Antes de corrigir, vou investigar a causa.

In [ ]:
brancos = df["TotalCharges"].astype(str).str.strip() == ""
print(f"Linhas com TotalCharges em branco: {brancos.sum()}")
df.loc[brancos, ["tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]

As 11 linhas em branco são todas de clientes com `tenure = 0`, ou seja, clientes que acabaram de entrar e ainda não fecharam o primeiro mês. O gasto acumulado delas é zero por lógica, não é um erro aleatório: é um vazio que tem significado.

Então a decisão é converter `TotalCharges` para número e preencher esses casos com 0. Preencher com a média seria um erro grave, porque colocaria milhares de reais de gasto em quem nunca pagou uma fatura.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Nulos após a conversão:", df["TotalCharges"].isna().sum())

df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("Tipo agora:", df["TotalCharges"].dtype)
print("Nulos restantes:", df["TotalCharges"].isna().sum())

Depois da conversão, `TotalCharges` virou uma coluna numérica (`float64`) sem nulos, e já entra nas estatísticas numéricas.

Preencher com 0 aqui não causa vazamento de dados. Esse 0 não saiu dos dados, saiu da lógica do problema: cliente com `tenure = 0` ainda não pagou nada, então o total acumulado é 0. Seria diferente se eu tivesse usado a média de `TotalCharges` para preencher. Média é estatística, teria que ser calculada só no treino, senão contaminaria a avaliação. Esse mesmo cuidado reaparece nas próximas etapas.

### 4.2 Removendo o identificador e preparando a variável-alvo

In [ ]:
df = df.drop(columns=["customerID"])

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

X = df.drop(columns=["Churn"])
y = df["Churn"]

print("Formato de X (preditores):", X.shape)
print("Formato de y (alvo):     ", y.shape)
print("\nDistribuição do alvo (0 = ficou, 1 = cancelou):")
print(y.value_counts())

Duas decisões aqui.

Removi o `customerID`. É só um código de identificação, não tem poder preditivo, e mantê-lo poderia até atrapalhar, porque o modelo tentaria decorar clientes em vez de aprender padrão.

Converti a variável-alvo `Churn` de texto (`Yes`/`No`) para número (`1`/`0`), que é o formato que os algoritmos exigem. A convenção que usei: 1 é quem cancelou, a classe de interesse, e 0 é quem permaneceu.

Por fim separei os dados em `X`, os atributos preditores, e `y`, o alvo.

### 4.3 Identificando variáveis numéricas e categóricas

Para tratar cada tipo de variável do jeito certo, separo as colunas numéricas das categóricas.

In [ ]:
num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Variáveis numéricas ({len(num_cols)}): {num_cols}")
print(f"\nVariáveis categóricas ({len(cat_cols)}): {cat_cols}")

Temos 4 variáveis numéricas (`SeniorCitizen`, `tenure`, `MonthlyCharges` e `TotalCharges`) e 15 categóricas, que são texto. Essa separação importa porque cada tipo pede um tratamento diferente na modelagem:

- As categóricas precisam ser codificadas em número, porque o modelo não entende `"Yes"` ou `"Fiber optic"`.
- As numéricas vão ser padronizadas, colocadas na mesma escala, o que ajuda modelos sensíveis a escala.

Esses dois tratamentos vão para dentro de um pipeline e são ajustados só no treino, nas próximas etapas. É assim que se evita o vazamento de dados.

## 5. Divisão dos dados em treino e teste

Para avaliar com honestidade se o modelo aprendeu, escondo uma parte dos dados (o teste) e treino só na outra (o treino). O modelo é avaliado apenas em dados que nunca viu. Assim eu meço se ele generaliza para clientes novos, e não se só decorou os exemplos.

### 5.1 Separando treino (70%) e teste (30%)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

print(f"Treino: {X_train.shape[0]} clientes | Teste: {X_test.shape[0]} clientes")
print(f"% de churn no treino: {y_train.mean() * 100:.1f}%")
print(f"% de churn no teste:  {y_test.mean() * 100:.1f}%")

Separei 70% para treino (4.930 clientes) e 30% para teste (2.113). O `stratify=y` garante que a proporção de churn de 26,5% seja a mesma nos dois conjuntos, o que importa num problema desbalanceado para o teste continuar representativo. O `random_state=SEED` deixa a divisão reprodutível.

### 5.2 Pré-processamento à prova de vazamento

Agora defino como as variáveis vão ser transformadas. Em vez de aplicar as transformações na mão, o que abriria brecha para vazamento, monto um `ColumnTransformer`, uma espécie de receita que roda dentro do pipeline de cada modelo e é ajustada só com os dados de treino.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
preprocessor

Esse desenho é o que impede o vazamento. O `preprocessor` acima é só a receita, ele ainda não olhou nenhum dado. Na próxima etapa ele entra no pipeline de cada modelo e é ajustado (`fit`) apenas com `X_train`. Assim, a média e o desvio da padronização, e as categorias aprendidas na codificação, saem exclusivamente do treino. O teste fica lacrado até a hora de avaliar.

O que cada transformação faz:

- `StandardScaler` (numéricas): coloca as variáveis na mesma escala, com média 0 e desvio 1. Sem isso, `TotalCharges`, na casa dos milhares, dominaria `tenure`, na casa das dezenas, em modelos sensíveis a escala como o KNN.
- `OneHotEncoder` (categóricas): cria uma coluna 0/1 para cada categoria. Por exemplo, `Contract` vira 3 colunas (`Month-to-month`, `One year`, `Two year`). É por isso que os 19 atributos originais viram 45 colunas.

## 6. Modelagem e avaliação

Seguindo o roteiro do MVP, treino um baseline, que é uma referência ingênua, e dois modelos candidatos, e comparo os resultados. Todos usam pipelines que aplicam o pré-processamento, ajustado só no treino, antes de treinar. É o que garante que não haja vazamento.

Sobre as métricas: como o problema é desbalanceado, a acurácia sozinha engana, então uso quatro.

| Métrica | O que responde | Por que importa aqui |
|---|---|---|
| Acurácia | % de acertos totais | Serve de referência, mas engana com desbalanceamento |
| Precisão | Dos previstos como "vai cancelar", quantos cancelaram? | Evita alarme falso, gastar retenção à toa |
| Recall | Dos que realmente cancelaram, quantos o modelo pegou? | A mais crítica: cliente que escapa é receita perdida |
| F1 | Equilíbrio entre precisão e recall | Resumo justo num cenário desbalanceado |

Para churn, o que priorizo é o recall e o F1 da classe que cancela. Deixar escapar quem ia cancelar (falso negativo) costuma sair mais caro do que abordar quem não ia sair (falso positivo).

Abaixo defino uma função de avaliação reutilizável, para medir todos os modelos do mesmo jeito.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)

resultados = []

def avaliar_modelo(nome, modelo):
    """Avalia um modelo treinado no conjunto de TESTE e guarda suas métricas."""
    y_pred = modelo.predict(X_test)
    metricas = {
        "Modelo": nome,
        "Acurácia": accuracy_score(y_test, y_pred),
        "Precisão": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
    }
    resultados.append(metricas)
    print(nome)
    for chave, valor in metricas.items():
        if chave != "Modelo":
            print(f"  {chave}: {valor:.3f}")
    return y_pred

### 6.1 Baseline: o modelo de referência

O baseline é a abordagem mais ingênua que existe: sempre prever a classe majoritária, o "não cancela", ignorando os atributos. Ele serve de régua. Qualquer modelo de verdade precisa superá-lo para justificar a própria complexidade.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import ConfusionMatrixDisplay

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

y_pred_base = avaliar_modelo("Baseline (sempre 'não cancela')", baseline)

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_base, display_labels=["Ficou", "Cancelou"], cmap="Blues"
)
plt.title("Matriz de confusão — Baseline")
plt.show()

O baseline expõe bem o problema. Ele acerta 73,5%, então a acurácia parece boa, mas tem recall 0 e F1 0: não identifica um único cliente que cancelou. A matriz de confusão confirma: ele prevê "ficou" para todos, acerta os 1.552 que ficaram e deixa escapar os 561 que cancelaram.

É a lição central do projeto. Num problema desbalanceado, acurácia alta pode significar um modelo inútil. Os próximos modelos precisam superar esse baseline onde importa: no recall e no F1 da classe que cancela, não só na acurácia.

### 6.2 Modelo candidato 1: Regressão Logística

A Regressão Logística é um modelo linear clássico para classificação binária: rápido, interpretável e um ótimo ponto de partida. Ela estima a probabilidade de o cliente cancelar a partir de uma combinação das variáveis. Uso um pipeline que junta o pré-processamento ao modelo, então o `fit` aplica a padronização e a codificação só no treino, sem vazamento.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe_lr = Pipeline([
    ("prep", preprocessor),
    ("modelo", LogisticRegression(max_iter=1000, random_state=SEED)),
])
pipe_lr.fit(X_train, y_train)

y_pred_lr = avaliar_modelo("Regressão Logística", pipe_lr)
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr, display_labels=["Ficou", "Cancelou"], cmap="Blues"
)
plt.title("Matriz de confusão — Regressão Logística")
plt.show()

A Regressão Logística melhorou bastante em relação ao baseline. O recall subiu de 0 para cerca de 0,56, então já detecta mais da metade dos cancelamentos, e o F1 chegou perto de 0,61. A acurácia também subiu, para cerca de 0,81. O modelo aprendeu de fato, não está apenas repetindo a classe majoritária como o baseline.

### 6.3 Modelo candidato 2: Random Forest

A Random Forest é um conjunto (ensemble) de várias árvores de decisão que votam juntas. Ela captura relações não-lineares e interações entre variáveis, e costuma ser robusta em dados tabulares. É bem diferente da Regressão Logística, e comparar dois modelos de famílias distintas dá uma visão mais rica.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

pipe_rf = Pipeline([
    ("prep", preprocessor),
    ("modelo", RandomForestClassifier(random_state=SEED, n_jobs=-1)),
])
pipe_rf.fit(X_train, y_train)

y_pred_rf = avaliar_modelo("Random Forest", pipe_rf)
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf, display_labels=["Ficou", "Cancelou"], cmap="Blues"
)
plt.title("Matriz de confusão — Random Forest")
plt.show()

A Random Forest também supera o baseline, mas aqui ficou um pouco atrás da Regressão Logística. Modelo mais complexo nem sempre vence: em dados tabulares com relações relativamente simples, um modelo linear bem ajustado costuma ser muito competitivo.

### 6.4 Comparação dos modelos

In [ ]:
df_resultados = pd.DataFrame(resultados).set_index("Modelo").round(3)
print(df_resultados)

ax = df_resultados[["Acurácia", "Precisão", "Recall", "F1"]].plot(
    kind="bar", figsize=(9, 5)
)
ax.set_title("Comparação dos modelos")
ax.set_ylabel("Pontuação (0 a 1)")
ax.set_ylim(0, 1)
plt.xticks(rotation=12, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

Lendo a comparação:

Os dois modelos superam o baseline com folga. Ambos saíram de recall 0 para detectar mais da metade dos cancelamentos, então de fato aprenderam padrão dos dados.

A Regressão Logística ficou à frente da Random Forest em todas as métricas, inclusive no F1 (0,61 contra 0,53) e no recall (0,56 contra 0,47). É um resultado comum em dados tabulares, com o modelo mais simples superando o mais complexo, e reforça que mais complexo não significa melhor.

Ainda assim, um recall de cerca de 56% significa que quase metade dos clientes que cancelam continua escapando. Há espaço para melhorar, e é o que faço na etapa de otimização de hiperparâmetros, buscando subir o recall sem comprometer a precisão.

## 7. Otimização de hiperparâmetros

Até aqui usei as configurações padrão dos modelos. Agora ajusto os hiperparâmetros, que são os botões de regulagem do modelo, da Regressão Logística, a melhor até agora, buscando melhorar o desempenho na classe que interessa, quem cancela.

O que ajustei e por quê:

- `C` controla a regularização, o quanto o modelo evita ficar complexo demais. Testei valores de 0,01 a 10.
- `class_weight` permite dar mais peso à classe minoritária. Como o problema é desbalanceado, o valor `"balanced"` pode ajudar o modelo a prestar mais atenção em quem cancela.

A estratégia foi Grid Search, que testa todas as combinações, com validação cruzada estratificada de 5 dobras. A validação cruzada divide o treino em 5 partes e testa cada combinação em rodadas diferentes, então eu escolho a melhor configuração sem nunca tocar no teste, o que de novo evita vazamento. Otimizei pelo F1, porque ele equilibra precisão e recall num cenário desbalanceado.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

grade_parametros = {
    "modelo__C": [0.01, 0.1, 1, 10],
    "modelo__class_weight": [None, "balanced"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

grid = GridSearchCV(pipe_lr, grade_parametros, scoring="f1", cv=cv, n_jobs=-1)
grid.fit(X_train, y_train)

print("Melhor combinação:", grid.best_params_)
print(f"Melhor F1 (média na validação cruzada): {grid.best_score_:.3f}")

A melhor combinação encontrada foi `C = 0.01`, que é uma regularização forte e deixa o modelo mais simples e conservador, com `class_weight = "balanced"`, que dá mais peso à classe minoritária. A validação cruzada avaliou cada combinação usando só os dados de treino, e o teste seguiu intocado. Agora aplico esse melhor modelo no teste.

In [ ]:
modelo_otimizado = grid.best_estimator_
y_pred_otim = avaliar_modelo("Regressão Logística (otimizada)", modelo_otimizado)

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_otim, display_labels=["Ficou", "Cancelou"], cmap="Blues"
)
plt.title("Matriz de confusão — Regressão Logística otimizada")
plt.show()

A otimização revelou um trade-off importante:

| Métrica | Log. Regressão (padrão) | Log. Regressão (otimizada) | Efeito |
|---|---|---|---|
| Acurácia | 0,809 | 0,743 | caiu |
| Precisão | 0,667 | 0,511 | caiu |
| Recall | 0,560 | 0,790 | subiu muito |
| F1 | 0,609 | 0,620 | subiu um pouco |

O `class_weight="balanced"` fez o modelo dar mais atenção à classe minoritária, quem cancela. O recall subiu de 56% para 79%: o modelo passa a detectar quase 4 em cada 5 clientes que vão cancelar, contra pouco mais da metade antes. O preço foi a queda na precisão, com mais alarmes falsos, e na acurácia.

Isso é melhora ou piora? Depende do objetivo, e é aqui que a análise vira decisão de negócio. Para retenção, esse trade-off costuma valer a pena: é melhor abordar alguns clientes que não iam sair (falso positivo, custo baixo, uma ligação ou um desconto) do que deixar escapar quem ia cancelar (falso negativo, custo alto, receita perdida). Como priorizo o recall e o F1, e os dois melhoraram, o modelo otimizado é a melhor solução para o objetivo, mesmo com a acurácia menor.

## 8. Avaliação final dos resultados

Aqui aprofundo a análise: verifico overfitting e underfitting, olho os erros do modelo escolhido e discuto limitações e melhorias.

### 8.1 Verificação de overfitting / underfitting

Um modelo que vai muito melhor no treino do que no teste está decorando em vez de aprendendo, o que é overfitting. Comparo o F1 nos dois conjuntos: quanto maior a diferença, maior o overfitting.

In [ ]:
from sklearn.metrics import f1_score

modelos_treinados = {
    "Regressão Logística": pipe_lr,
    "Random Forest": pipe_rf,
    "Log. Regressão (otimizada)": modelo_otimizado,
}

linhas = []
for nome, modelo in modelos_treinados.items():
    f1_treino = f1_score(y_train, modelo.predict(X_train))
    f1_teste = f1_score(y_test, modelo.predict(X_test))
    linhas.append({
        "Modelo": nome,
        "F1 treino": round(f1_treino, 3),
        "F1 teste": round(f1_teste, 3),
        "Diferença": round(f1_treino - f1_teste, 3),
    })

pd.DataFrame(linhas).set_index("Modelo")

Diagnóstico de overfitting:

A Regressão Logística, tanto padrão quanto otimizada, teve desempenho quase igual no treino e no teste. Isso indica que ela generaliza bem, aprendeu o padrão e não os exemplos.

A Random Forest teve F1 de quase 1,00 no treino contra cerca de 0,53 no teste, uma diferença enorme. É um caso claro de overfitting: ela praticamente decorou o treino e não sabe lidar com cliente novo. Isso explica por que ficou atrás na comparação. Dava para atenuar limitando a profundidade das árvores, mas como a Regressão Logística já resolve bem e é mais simples e interpretável, segui com ela.

Nenhum modelo teve underfitting grave. O baseline, por design, sub-ajusta, já que não aprende nada.

### 8.2 Análise de erros do modelo escolhido

O modelo escolhido é a Regressão Logística otimizada, por ter o melhor equilíbrio para o objetivo, com o maior recall e o melhor F1. Olhando a matriz de confusão dele no teste, com 2.113 clientes:

- 443 dos 561 clientes que realmente cancelaram foram identificados (recall 79%).
- 118 canceladores escaparam (falsos negativos): o modelo disse que ficariam, mas cancelaram. É o erro mais custoso para o negócio.
- 424 falsos positivos: clientes sinalizados como risco que na verdade ficariam. Custo baixo, no máximo uma ação de retenção desnecessária.

Ou seja, o modelo erra mais para o lado seguro, prefere sinalizar demais a deixar escapar, o que combina com o objetivo de retenção. A empresa pode ajustar esse equilíbrio conforme o orçamento que tiver para campanhas.

### 8.3 Limitações e melhorias futuras

Limitações da solução:

- O recall de cerca de 79% ainda deixa perto de 21% dos cancelamentos passarem sem detecção.
- O ganho de recall veio com queda de precisão, com mais alarmes falsos: a empresa gastaria retenção com alguns clientes que não iam sair.
- O dataset é fictício e estático, sem histórico temporal, então os padrões tendem a ser mais limpos do que na realidade.
- Não há o valor financeiro de cada cliente, então otimizei por F1, e não pelo lucro real da campanha de retenção.

Melhorias possíveis mais adiante:

- Testar modelos de boosting, como Gradient Boosting e XGBoost, que costumam ir bem em dados tabulares.
- Calibrar o limiar de decisão (threshold) para ajustar o equilíbrio entre recall e precisão conforme o orçamento.
- Fazer engenharia de atributos, por exemplo combinar serviços contratados ou criar faixas de `tenure`.
- Incorporar novos dados, como reclamações, satisfação e uso do serviço.
- Otimizar diretamente por uma métrica de negócio, o retorno financeiro, em vez de apenas estatística.

## 9. Conclusão

O objetivo deste MVP foi prever o cancelamento de clientes (churn) de uma operadora de telecom, um problema de classificação binária com forte valor de negócio, já que reter um cliente costuma custar bem menos do que conquistar um novo.

Usei o dataset público Telco Customer Churn, da IBM, com 7.043 clientes e 20 atributos preditores. A variável-alvo, `Churn`, é desbalanceada: só 26,5% dos clientes cancelaram.

No tratamento, corrigi a coluna `TotalCharges`, que veio como texto e tinha 11 valores em branco de clientes recém-chegados, preenchidos com 0; removi o identificador `customerID`; e montei um pipeline de pré-processamento (padronização das numéricas e codificação one-hot das categóricas) ajustado só no treino, para evitar vazamento. A divisão foi 70/30 estratificada.

Comparei um baseline (sempre prever "não cancela"), uma Regressão Logística e uma Random Forest, e depois otimizei a Regressão Logística com Grid Search e validação cruzada.

A Regressão Logística otimizada (`C=0.01`, `class_weight="balanced"`) foi a escolhida. Ela chegou a recall de 79% e ao melhor F1, 0,62. Escolhi ela porque, no contexto de retenção, detectar o máximo de clientes em risco (recall alto) vale mais do que maximizar a acurácia. E ela faz isso sem sinal de overfitting, com desempenho parecido no treino e no teste, além de ser simples e interpretável.

Por que ela e não a Random Forest? A Random Forest, apesar de mais complexa, teve overfitting severo, quase perfeita no treino e fraca no teste, e ficou atrás nas métricas que interessam.

Cumpri o objetivo? Sim. Saí de um baseline inútil, com recall 0, para um modelo que identifica cerca de 4 em cada 5 clientes prestes a cancelar. É uma ferramenta que permitiria à empresa agir antes e reduzir perdas.

Como próximos passos, é possível testar modelos de boosting, calibrar o limiar de decisão conforme o orçamento de retenção, fazer engenharia de atributos e, no ideal, otimizar por uma métrica de retorno financeiro em vez de só estatística.

## 10. Boas práticas e checklist do MVP

Esta seção registra as boas práticas que adotei e responde, de forma objetiva, às perguntas do checklist sugerido pelo MVP.

### 10.1 Bibliotecas utilizadas e reprodutibilidade

Bibliotecas: `pandas` e `numpy` para manipular os dados, `matplotlib` e `seaborn` para visualização, e `scikit-learn` para o pré-processamento, os modelos, as métricas e a otimização. Todas já vêm instaladas no Google Colab, então não é preciso instalar nada.

Reprodutibilidade: fixei uma semente (`SEED = 42`) usada na divisão treino/teste, na validação cruzada e nos modelos, o que faz qualquer execução cair nos mesmos resultados. Os dados vêm de uma URL pública, então o notebook roda do começo ao fim sem configuração manual.

Recursos: o treino é leve, questão de segundos numa CPU comum, sem GPU nem hardware especial.

### 10.2 Respostas ao checklist do MVP

**Definição do problema**
- *Descrição do problema:* prever o cancelamento (churn) de clientes de uma operadora de telecom.
- *Objetivo do modelo:* identificar clientes com alto risco de cancelar, para permitir ações de retenção.
- *Tipo de problema:* classificação binária supervisionada.
- *Por que ML:* há exemplos históricos rotulados (cancelou/ficou); o modelo aprende o padrão e prevê novos casos.
- *Premissas/hipóteses:* dados históricos representam o comportamento futuro; contrato, tempo de casa e forma de pagamento influenciam o churn.
- *Restrições:* usamos apenas os atributos do dataset; base pública e não utilizada nas aulas.

**Descrição dos dados**
- *Dataset e fonte:* Telco Customer Churn (IBM, base pública e fictícia).
- *Como foi carregado:* diretamente de uma URL `raw` pública do GitHub, sem upload/login.
- *Registros e atributos:* 7.043 clientes × 21 colunas (20 preditores + alvo).
- *Principais atributos:* `Contract`, `tenure`, `PaymentMethod`, `InternetService`, `MonthlyCharges`, `TotalCharges`.
- *Variável-alvo:* `Churn` (Yes/No → 1/0).
- *Limitações:* dados fictícios, estáticos, sem histórico de uso ou satisfação.

**Preparação e divisão**
- *Valores ausentes:* 11 em `TotalCharges` (clientes com `tenure=0`) → preenchidos com 0 (regra de negócio).
- *Remoção/transformação:* `customerID` removido; alvo convertido para 0/1.
- *Codificação/normalização:* one-hot nas categóricas + padronização nas numéricas, dentro de um pipeline.
- *Vazamento de dados:* evitado, pré-processamento ajustado só no treino.
- *Divisão:* 70% treino / 30% teste, estratificada; validação cruzada usada na otimização.

**Modelagem, otimização e avaliação**
- *Baseline:* `DummyClassifier` (sempre "não cancela"), acurácia 0,735, recall 0.
- *Modelos treinados:* Regressão Logística e Random Forest (ambos além do baseline).
- *Comparação justa:* mesmo pipeline e mesmo conjunto de teste para todos.
- *Underfitting/overfitting:* Random Forest apresentou overfitting (F1 treino ~1,0 × teste ~0,53); a Regressão Logística generalizou bem.
- *Otimização:* Grid Search + validação cruzada (5 dobras) ajustando `C` e `class_weight` da Regressão Logística; recall subiu de 0,56 para 0,79.
- *Métricas e por quê:* acurácia, precisão, recall, F1 e matriz de confusão, com foco em recall/F1 por causa do desbalanceamento.
- *Melhor modelo:* Regressão Logística otimizada (melhor recall e F1, sem overfitting, interpretável).

**Conclusão**
- *Melhor solução e por quê:* Regressão Logística otimizada, melhor equilíbrio para o objetivo de retenção.
- *Cumpriu o objetivo?* Sim: de recall 0 (baseline) para 79%.
- *Próximos passos:* boosting, calibração de limiar, engenharia de atributos, otimização por retorno financeiro.